# El agente sobre la copia seudonimizada (P1-b.2b)
El agente y el toolkit consultan **la copia**: lo que recibe el modelo sale en alias (`U-0032`, `IP-0042`). El ledger se abre con los
datos **reales** y registra sobre qué copia corrió cada consulta; el `replay` re-ejecuta cada una sobre la suya. Ver `docs/privacidad.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import pandas as pd

from dfir_copilot.agent.graph import build_agent
from dfir_copilot.agent.llm import make_llm
from dfir_copilot.cases import CaseWorkspace
from dfir_copilot.privacy import detector_parity

ws = CaseWorkspace.open("IDOR-INVOICES-2020Q4-TZ-SANTIAGO")
real = ws.engine(timeout_s=120)                  # datos reales: solo para el analista y para el ledger
engine, ps = ws.pseudonymized(timeout_s=120)     # copia (la crea la primera vez) + diccionario LOCAL
ledger = ws.ledger(real)                         # el ledger se ata siempre al motor real
print("real :", real.copy_id)
print("copia:", engine.copy_id)

## 1. Paridad: ¿la copia da las mismas conclusiones que los datos reales?
Los detectores que ve el modelo corren sobre la copia. `scrub` y `mask_*` pueden fundir valores distintos en uno; esto lo mide con
tus datos reales. El resultado solo lleva alias y cifras (las diferencias de texto, solo el nombre de la métrica).

In [ ]:
parity = detector_parity(real, engine, ps)
print("equivalente:", parity["equivalent"])
display(pd.DataFrame([{"detector": n, "real": d["findings"]["real"], "copia": d["findings"]["pseudo"], "coinciden": d["matched"],
                       "solo_real": len(d["only_real"]), "solo_copia": len(d["only_pseudo"]),
                       "dif_numéricas": len(d["numeric_differences"]), "dif_texto": len(d["text_differences"]),
                       "equivalente": d["equivalent"]} for n, d in parity["detectors"].items()]))

## 2. Investigar con el agente

In [ ]:
agent = build_agent(engine, ledger, make_llm(), max_steps=12)   # con `real` en lugar de `engine` lanzaría PrivacyError
n0 = len(ledger.entries())
r = agent.ask("¿Hay actividad anómala sobre /invoices/search? Investiga el dataset, formula hipótesis y pruébalas.")
print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens)
print(r.answer or "(pausado: hay hipótesis pendientes de tu aprobación)")

In [ ]:
# Lo que está en el ledger y lo que vio el modelo va en alias; esto lo traduce SOLO para ti, en local.
if r.answer:
    print(ps.reveal_any(r.answer))
for req in agent.pending():
    print(f"{req['hypothesis_id']}: {req['estado_actual']} -> {req['estado_solicitado']}")
    print("  Hipótesis:    ", ps.reveal_any(req["hipotesis"]))
    print("  Justificación:", ps.reveal_any(req["justificacion"]))

Al aprobar o rechazar, **escribe la nota con alias**: lo que escribes en `resolve()` y en `ask()` viaja al modelo tal cual (la
traducción automática del texto del analista queda para P1-b.3).

In [ ]:
# r = agent.resolve("approve", "Revisé la evidencia citada con alias U-0032…U-0035; coincide con mi análisis.")
# print(r.status, r.answer)

## 3. Qué quedó registrado y sobre qué copia

In [ ]:
copies = pd.Series([e["data"].get("copy", "real (sin sello)") for e in ledger.entries("query")]).value_counts()
display(copies.rename("consultas").to_frame())
for e in ledger.entries("data_copy"):
    d = e["data"]
    print(e["seq"], d["copy_id"], "| política", d["policy"]["version"], "| filas", d["rows"], "| diccionario", d["aliases_sha256"][:12])
print(ledger.verify())

## 4. Replay copia por copia
Con **los dos motores** se verifica todo. Con uno solo, las consultas de la otra copia salen como `skipped` (no verificadas, que no es
lo mismo que alteradas).

In [ ]:
rep = ledger.replay([real, engine])
ok = [x for x in rep if x["match"]]
print(len(rep), "consultas |", len(ok), "coinciden |", sum(bool(x.get("skipped")) for x in rep), "sin motor")
display(pd.DataFrame(rep).groupby("copy")["match"].agg(["count", "sum"]))

In [ ]:
bad = [x for x in rep if not x["match"]]
display(pd.DataFrame(bad).reindex(columns=["query_id", "copy", "recorded_rows", "replayed_rows", "hash_match", "error"]))
by_id = {e["data"]["query_id"]: e["data"] for e in ledger.entries("query")}
for x in bad:
    d = by_id[x["query_id"]]
    print(x["query_id"], "| ejecutada", d["executed_at_utc"], "| limit", d.get("limit"),
          "| truncated", d.get("truncated"), "| con huella:", d.get("result_sha256") is not None)
print("\nreplays anteriores:")
for e in ledger.entries("replay"):
    print(e["seq"], e["ts_utc"], {k: e["data"][k] for k in ("queries", "matches", "mismatches") if k in e["data"]})

In [ ]:
[ps.reveal("src_ip", f"IP-{n}") for n in range(1806, 1811)]

In [ ]:
for x in bad:
    print(x["query_id"], "|", by_id[x["query_id"]]["sql"][:120])
cols = [r[0] for r in real.query("DESCRIBE logs").rows]
print(len(cols), "columnas ahora | zona local:", real.local_timezone)
print("timestamp_local" in cols)

In [ ]:
ledger.note(
    "Las 4 consultas DESCRIBE logs (una por detector) no reproducen: 17 columnas registradas, 18 ahora. La vista del motor "
    "añade timestamp_local con la zona declarada. El Parquet no cambió (ledger.verify OK). No es alteración de datos.",
    refs=[x["query_id"] for x in bad], analyst="Eder")
print(ledger.verify())